# 03 — Model Selection & Sanity Tests

Bu notebook, tezde kullanılacak ana LLM'leri küçük ve sabit bir **development sanity subset**
üzerinde karşılaştırır.

Amaç:
- Türkçe kısa factual QA kalitesini görmek
- Colab/T4 + 4-bit uyumluluğunu test etmek
- deterministic generation davranışını kontrol etmek
- token log-probability ve entropy çıkarılabildiğini doğrulamak
- inference süresini ölçmek
- iki ana modeli seçip manifest ile dondurmak

**Metodolojik kural:** model-selection soruları daha sonra final held-out test setine girmemelidir.

Bu notebook yalnız `validated_vXXX` dataset okur; living master veya raw dataset kullanmaz.

## 1. Drive ve bağımlılıklar

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
!pip -q install "transformers==4.57.6" "accelerate>=1.3,<2" "bitsandbytes>=0.45" openpyxl "huggingface_hub>=0.34"

## 2. Imports ve runtime

In [ ]:
import gc, json, re, time, hashlib, platform
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import transformers
import accelerate
import bitsandbytes as bnb

if not torch.cuda.is_available():
    raise RuntimeError("Colab GPU runtime açılmalıdır.")

print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Accelerate:", accelerate.__version__)
print("bitsandbytes:", bnb.__version__)

## 3. Dataset ve deney config

In [ ]:
PROJECT_ROOT = Path("/content/drive/MyDrive/LLM_Confidence_Thesis")

DATA_VALIDATED_DIR = PROJECT_ROOT / "data" / "validated"
SPLITS_DIR = PROJECT_ROOT / "data" / "splits" / "model_selection"
RUNS_DIR = PROJECT_ROOT / "runs" / "model_selection"
RESULTS_DIR = PROJECT_ROOT / "results" / "model_selection"

for p in [SPLITS_DIR, RUNS_DIR, RESULTS_DIR]:
    p.mkdir(parents=True, exist_ok=True)

# ============================================================
VALIDATED_DATASET_VERSION = "validated_v002"
# ============================================================

SANITY_N = 30
SANITY_SEED = 3407

PROMPT_VERSION = "short_fact_tr_v1"
GENERATION_CONFIG_VERSION = "deterministic_v2_model_specific_eos"
MAX_NEW_TOKENS = 20

DATASET_JSON_PATH = DATA_VALIDATED_DIR / f"{VALIDATED_DATASET_VERSION}.json"
DATASET_MANIFEST_PATH = DATA_VALIDATED_DIR / f"{VALIDATED_DATASET_VERSION}_manifest.json"

print(DATASET_JSON_PATH)

## 4. Validated dataset checksum doğrulaması

In [ ]:
def sha256_file(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

if not DATASET_JSON_PATH.exists():
    raise FileNotFoundError(
        f"{DATASET_JSON_PATH}\nÖnce 02 notebook ile bu sürümü tamamen validate edin."
    )

with open(DATASET_JSON_PATH, "r", encoding="utf-8") as f:
    validated_data = json.load(f)

with open(DATASET_MANIFEST_PATH, "r", encoding="utf-8") as f:
    dataset_manifest = json.load(f)

dataset_sha256 = sha256_file(DATASET_JSON_PATH)

if dataset_sha256 != dataset_manifest["validated_file_sha256"]:
    raise RuntimeError("Validated dataset checksum uyuşmuyor.")

if len(validated_data) != dataset_manifest["n_validated_questions"]:
    raise RuntimeError("Dataset soru sayısı manifest ile uyuşmuyor.")

df = pd.DataFrame(validated_data)

if df["question_id"].duplicated().any():
    raise RuntimeError("Duplicate question_id var.")

if not (df["verification_status"] == "verified").all():
    raise RuntimeError("Model selection yalnız verified sorularla yapılmalıdır.")

print("DATASET OK")
print("Version:", VALIDATED_DATASET_VERSION)
print("N:", len(df))
print("SHA-256:", dataset_sha256)

## 5. Immutable sanity subset

30 soruluk subset ilk çalıştırmada oluşturulur ve sonra değiştirilmez.
Answer type dağılımını yaklaşık korur.

Bu subset'in question ID'leri manifestte
`must_be_excluded_from_final_test = true` olarak kaydedilir.

In [ ]:
SANITY_PATH = SPLITS_DIR / (
    f"{VALIDATED_DATASET_VERSION}_sanity_n{SANITY_N}_seed{SANITY_SEED}.json"
)
SANITY_MANIFEST_PATH = SANITY_PATH.with_name(
    SANITY_PATH.stem + "_manifest.json"
)

def make_stratified_subset(dataframe, n, seed):
    if n > len(dataframe):
        raise ValueError("SANITY_N dataset boyutundan büyük.")

    rng = np.random.default_rng(seed)
    groups = dataframe.groupby("answer_type")
    selected = []

    # Her answer type'tan en az bir örnek.
    for _, group in groups:
        selected.append(
            group.sample(
                n=1,
                random_state=int(rng.integers(0, 2**31 - 1))
            )
        )

    base = pd.concat(selected)
    remaining_n = n - len(base)

    if remaining_n > 0:
        remaining = dataframe.drop(index=base.index)
        extra = remaining.sample(
            n=remaining_n,
            random_state=seed,
        )
        base = pd.concat([base, extra])

    return base.sample(
        frac=1,
        random_state=seed,
    ).reset_index(drop=True)

if SANITY_PATH.exists():
    with open(SANITY_PATH, "r", encoding="utf-8") as f:
        sanity_data = json.load(f)
    sanity_df = pd.DataFrame(sanity_data)
    print("Mevcut immutable sanity subset kullanılıyor.")
else:
    sanity_df = make_stratified_subset(df, SANITY_N, SANITY_SEED)
    sanity_data = sanity_df.to_dict("records")

    with open(SANITY_PATH, "x", encoding="utf-8") as f:
        json.dump(sanity_data, f, ensure_ascii=False, indent=2)

    manifest = {
        "purpose": "model_selection_development_only",
        "source_dataset_version": VALIDATED_DATASET_VERSION,
        "source_dataset_sha256": dataset_sha256,
        "n_questions": len(sanity_df),
        "seed": SANITY_SEED,
        "question_ids": sanity_df["question_id"].tolist(),
        "must_be_excluded_from_final_test": True,
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
    }
    with open(SANITY_MANIFEST_PATH, "x", encoding="utf-8") as f:
        json.dump(manifest, f, ensure_ascii=False, indent=2)

print("Sanity N:", len(sanity_df))
display(sanity_df["answer_type"].value_counts().rename("count").to_frame())
display(sanity_df[["question_id", "question", "answer_type", "category"]].head())

## 6. Aday modeller

İlk turda iki ana aday:
- `Qwen/Qwen3-4B`
- `microsoft/Phi-4-mini-instruct`

Üçüncü aday gerekiyorsa daha sonra listeye eklenebilir.

In [ ]:
CANDIDATE_MODELS = [
    {
        "model_id": "Qwen/Qwen3-4B",
        "short_name": "qwen3_4b",
        "revision": "3101254bbe4169895668a0e7653c3fd1f313576e",
        "trust_remote_code": False,
        "disable_thinking": True,
        "backend": "causal_lm",
        "gated": False,
        "compute_dtype": "float16",
        "model_dtype": "float16",
        "run_precision_tag": "",
    },
    {
        "model_id": "microsoft/Phi-4-mini-instruct",
        "short_name": "phi4_mini_instruct",
        "revision": "cfbefacb99257ffa30c83adab238a50856ac3083",
        "trust_remote_code": False,
        "disable_thinking": False,
        "backend": "causal_lm",
        "gated": False,
        "compute_dtype": "float16",
        "model_dtype": "float16",
        "run_precision_tag": "",
    },
    {
        "model_id": "google/gemma-3-4b-it",
        "short_name": "gemma3_4b_it",
        "revision": None,
        "trust_remote_code": False,
        "disable_thinking": False,
        "backend": "gemma3_processor",
        "gated": True,
        "compute_dtype": "float32",
        "model_dtype": "float32",
        "run_precision_tag": "_nf4fp32",
    },
]

display(pd.DataFrame(CANDIDATE_MODELS))

### Gemma erişimi ve revision sabitleme

`google/gemma-3-4b-it` gated bir Hugging Face repository'sidir.

İlk kullanım öncesinde:

1. Hugging Face hesabında Gemma kullanım koşullarını kabul edin.
2. Aşağıdaki login hücresini çalıştırın ve **read** yetkili Hugging Face token'ınızı girin.
3. Token'ı notebook içine düz metin olarak yazmayın.

Gemma'nın `main` branch'i doğrudan deney kimliği olarak kullanılmaz. Notebook, repository'nin o
andaki commit SHA'sını Hugging Face API'den çözer ve `candidate["revision"]` içine yazar.
Model yükleme ve run manifesti bundan sonra bu exact SHA ile yapılır.

In [ ]:
from huggingface_hub import login, HfApi

# ------------------------------------------------------------
# GEMMA İÇİN İLK KEZ / YENİ RUNTIME'DA GEREKİR
# ------------------------------------------------------------
# Aşağıdaki satır token istemi açar.
# Token'ı kaynak koda yazmayın.
login()

hf_api = HfApi()

def resolve_candidate_revisions(candidates):
    resolved = []

    for candidate in candidates:
        item = dict(candidate)

        if item["revision"] is None:
            info = hf_api.model_info(
                item["model_id"]
            )

            if not info.sha:
                raise RuntimeError(
                    f"Commit SHA çözülemedi: {item['model_id']}"
                )

            item["revision"] = info.sha

        resolved.append(item)

    return resolved


CANDIDATE_MODELS = resolve_candidate_revisions(
    CANDIDATE_MODELS
)

print("DONDURULMUŞ MODEL REVISION'LARI")
for candidate in CANDIDATE_MODELS:
    print(
        candidate["model_id"],
        "->",
        candidate["revision"],
    )

### Phi-4-mini yükleme notu

Phi-4-mini-instruct için `trust_remote_code=False` kullanılır. Phi-4 ailesi
Transformers'a native olarak entegre olduğundan Hub'daki `modeling_phi3.py`
dosyasını çalıştırmamız gerekmez.

Ayrıca her iki model için de repository `revision` commit hash'i sabitlenmiştir.
Bu, model weights/config/tokenizer sürümünün deneyler boyunca değişmemesini sağlar.

## 7. Prompt ve normalize yardımcıları

In [ ]:
import unicodedata

SYSTEM_PROMPT = (
    "Türkçe kısa ve doğrudan olgusal cevap ver. "
    "Sadece cevabı yaz; açıklama veya gerekçe verme. "
    "Cevabı bilmiyorsan yalnızca 'Bilmiyorum.' yaz."
)

def render_prompt(tokenizer, question, disable_thinking=False):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]

    if disable_thinking:
        try:
            return tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
                enable_thinking=False,
            )
        except TypeError:
            pass

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

def normalize_answer(text):
    if not isinstance(text, str):
        return ""

    text = unicodedata.normalize("NFKC", text)
    text = text.replace("İ", "i").replace("I", "ı")
    text = text.lower().strip()

    text = re.sub(
        r"^[\s\.,;:!?\"'`()\[\]{}]+|[\s\.,;:!?\"'`()\[\]{}]+$",
        "",
        text,
    )
    text = re.sub(r"\s+", " ", text).strip()

    return text

def auto_alias_match(answer, aliases):
    if not isinstance(aliases, list):
        return 0

    answer_norm = normalize_answer(answer)

    alias_norms = {
        normalize_answer(alias)
        for alias in aliases
        if isinstance(alias, str)
    }

    return int(answer_norm in alias_norms)


# ------------------------------------------------------------------
# Unknown / uncertainty output taxonomy
# ------------------------------------------------------------------

PURE_UNKNOWN_FORMS = {
    "bilmiyorum",
    "emin değilim",
    "bilinmiyor",
    "bilemiyorum",
}

UNKNOWN_MARKER_PATTERNS = [
    r"\bbilmiyorum\b",
    r"\bemin değilim\b",
    r"\bbilinmiyor\b",
    r"\bbilemiyorum\b",
]


def pure_unknown_response(answer):
    """
    Model yalnızca bir unknown/refusal cevabı mı verdi?

    Örnek:
      'Bilmiyorum.' -> 1
      'Canberra. Bilmiyorum.' -> 0
    """
    normalized = normalize_answer(answer)

    return int(
        normalized in PURE_UNKNOWN_FORMS
    )


def contains_unknown_marker(answer):
    """
    Cevabın herhangi bir yerinde explicit unknown marker var mı?

    Örnek:
      'Bilmiyorum.' -> 1
      'Canberra. Bilmiyorum.' -> 1
      'Canberra.' -> 0
    """
    normalized = normalize_answer(answer)

    return int(
        any(
            re.search(pattern, normalized)
            for pattern in UNKNOWN_MARKER_PATTERNS
        )
    )


def mixed_answer_unknown(answer):
    """
    Aynı output içinde hem substantive cevap hem explicit unknown marker var mı?

    Örnek:
      'Canberra. Bilmiyorum.' -> 1
      'Bilmiyorum.' -> 0
      'Canberra.' -> 0
    """
    contains = contains_unknown_marker(answer)
    pure = pure_unknown_response(answer)

    return int(
        contains == 1 and pure == 0
    )


def unknown_response(answer):
    """
    Backward-compatible ana unknown feature.

    Ana feature olarak yalnız pure unknown kabul edilir.
    Mixed output ayrı feature olarak tutulur.
    """
    return pure_unknown_response(answer)

### Gemma backend notu

Gemma 3'ün 4B checkpoint'i multimodal modeldir. Bu tezde yalnız text input kullanılmasına rağmen
modelin resmi Transformers yolu korunur:

```text
AutoProcessor
+
Gemma3ForConditionalGeneration
```

Qwen ve Phi ise:

```text
AutoTokenizer
+
AutoModelForCausalLM
```

yolunu kullanır.

Bu fark yalnız model-specific input plumbing'dir. Üç model için deney contract'ı aynıdır:
aynı 30 soru, aynı system instruction, deterministic generation, aynı `max_new_tokens` ve aynı
token-level logprob/entropy hesaplama yöntemi.

## 8. 4-bit model yükleme ve GPU temizleme

In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    AutoProcessor,
    Gemma3ForConditionalGeneration,
    BitsAndBytesConfig,
)

DTYPE_MAP = {
    "float16": torch.float16,
    "bfloat16": torch.bfloat16,
    "float32": torch.float32,
}

def candidate_compute_dtype(candidate):
    return DTYPE_MAP[candidate["compute_dtype"]]

def candidate_model_dtype(candidate):
    return DTYPE_MAP[candidate["model_dtype"]]

def quant_config(candidate):
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=candidate_compute_dtype(candidate),
    )

def clear_gpu():
    gc.collect()
    torch.cuda.empty_cache()

def get_tokenizer(frontend):
    if hasattr(frontend, "tokenizer"):
        return frontend.tokenizer
    return frontend

def load_model(candidate):
    clear_gpu()

    model_id = candidate["model_id"]
    revision = candidate["revision"]
    backend = candidate["backend"]
    model_dtype = candidate_model_dtype(candidate)
    compute_dtype = candidate_compute_dtype(candidate)

    print("Model:", model_id)
    print("Revision:", revision)
    print("Backend:", backend)
    print("Model dtype:", model_dtype)
    print("4-bit compute dtype:", compute_dtype)

    if backend == "gemma3_processor":
        frontend = AutoProcessor.from_pretrained(
            model_id,
            revision=revision,
            trust_remote_code=False,
        )

        model = Gemma3ForConditionalGeneration.from_pretrained(
            model_id,
            revision=revision,
            quantization_config=quant_config(candidate),
            device_map="auto",
            torch_dtype=model_dtype,
        )

    elif backend == "causal_lm":
        frontend = AutoTokenizer.from_pretrained(
            model_id,
            revision=revision,
            trust_remote_code=candidate["trust_remote_code"],
            use_fast=True,
        )

        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            revision=revision,
            quantization_config=quant_config(candidate),
            device_map="auto",
            trust_remote_code=candidate["trust_remote_code"],
            torch_dtype=model_dtype,
        )
    else:
        raise ValueError(f"Bilinmeyen backend: {backend}")

    model.eval()
    tokenizer = get_tokenizer(frontend)

    if tokenizer.pad_token_id is None:
        tokenizer.pad_token_id = tokenizer.eos_token_id

    print("Loaded model class:", model.__class__.__name__)
    print("Frontend class:", frontend.__class__.__name__)
    print("Tokenizer class:", tokenizer.__class__.__name__)
    print("Tokenizer EOS token:", tokenizer.eos_token)
    print("Tokenizer EOS id:", tokenizer.eos_token_id)
    print("Model generation EOS id(s):", model.generation_config.eos_token_id)

    return frontend, model

### Generation stop-token düzeltmesi

`generate()` içinde tokenizer'ın tek EOS ID'si zorlanmaz. Modelin kendi `generation_config.eos_token_id` değeri kullanılır. Bu özellikle birden fazla turn-ending token kullanan Phi-4-mini gibi modellerde cevap bittikten sonra yeni sahte konuşma üretimini engeller.

## 9. Tek soru inference + logprob/entropy probe

In [ ]:
def build_model_inputs(
    frontend,
    model,
    candidate,
    question,
):
    backend = candidate["backend"]

    if backend == "gemma3_processor":
        messages = [
            {
                "role": "system",
                "content": [
                    {
                        "type": "text",
                        "text": SYSTEM_PROMPT,
                    }
                ],
            },
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": question,
                    }
                ],
            },
        ]

        inputs = frontend.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt",
        )

        inputs = {
            key: value.to(model.device)
            for key, value in inputs.items()
        }

        return inputs

    tokenizer = get_tokenizer(frontend)

    prompt = render_prompt(
        tokenizer,
        question,
        disable_thinking=candidate[
            "disable_thinking"
        ],
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
    )

    return {
        key: value.to(model.device)
        for key, value in inputs.items()
    }


def decode_generated(
    frontend,
    generated_ids,
):
    if hasattr(frontend, "decode"):
        return frontend.decode(
            generated_ids,
            skip_special_tokens=True,
        ).strip()

    tokenizer = get_tokenizer(frontend)

    return tokenizer.decode(
        generated_ids,
        skip_special_tokens=True,
    ).strip()


@torch.inference_mode()
def run_one(
    frontend,
    model,
    candidate,
    question,
):
    inputs = build_model_inputs(
        frontend,
        model,
        candidate,
        question,
    )

    prompt_len = inputs[
        "input_ids"
    ].shape[1]

    torch.cuda.synchronize()
    t0 = time.perf_counter()

    # EOS ID'yi burada özellikle override etmiyoruz.
    # Her model kendi generation_config değerini kullanır.
    out = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        return_dict_in_generate=True,
        output_scores=True,
    )

    torch.cuda.synchronize()
    elapsed = (
        time.perf_counter()
        - t0
    )

    generated_ids = out.sequences[0][
        prompt_len:
    ]

    answer = decode_generated(
        frontend,
        generated_ids,
    )

    token_logprobs = []
    token_entropies = []

    for step, logits_batch in enumerate(
        out.scores
    ):
        if step >= len(generated_ids):
            break

        logits = logits_batch[
            0
        ].float()

        log_probs = torch.log_softmax(
            logits,
            dim=-1,
        )

        probs = torch.softmax(
            logits,
            dim=-1,
        )

        token_id = int(
            generated_ids[
                step
            ].item()
        )

        token_logprobs.append(
            float(
                log_probs[
                    token_id
                ].item()
            )
        )

        token_entropies.append(
            float(
                -(
                    probs
                    *
                    log_probs
                ).sum().item()
            )
        )

    return {
        "answer": answer,
        "content_token_count": int(
            len(generated_ids)
        ),
        "generation_seconds": elapsed,
        "mean_logprob_probe": (
            float(
                np.mean(
                    token_logprobs
                )
            )
            if token_logprobs
            else np.nan
        ),
        "min_logprob_probe": (
            float(
                np.min(
                    token_logprobs
                )
            )
            if token_logprobs
            else np.nan
        ),
        "mean_token_entropy_probe": (
            float(
                np.mean(
                    token_entropies
                )
            )
            if token_entropies
            else np.nan
        ),
        "logprob_probe_available": int(
            bool(token_logprobs)
        ),
        "entropy_probe_available": int(
            bool(token_entropies)
        ),
    }

## 9B. Gemma FP32 smoke test

30 soruluk run'dan önce tek soru ile kontrol edin.

Gemma 3'te FP16 generation boş output üretebildiği için bu sürümde Gemma:

```text
NF4 weights + FP32 compute
```

ile çalışır.

Beklenen: `Answer:` alanı boş olmamalıdır.

In [ ]:
gemma_candidate = CANDIDATE_MODELS[2]

print("CUDA BF16 supported:", torch.cuda.is_bf16_supported())

frontend_smoke, model_smoke = load_model(gemma_candidate)

smoke_row = sanity_df.iloc[0]
smoke_result = run_one(
    frontend_smoke,
    model_smoke,
    gemma_candidate,
    smoke_row["question"],
)

print("Question:", smoke_row["question"])
print("Answer:", repr(smoke_result["answer"]))
print("Generated tokens:", smoke_result["content_token_count"])

if not smoke_result["answer"].strip():
    raise RuntimeError(
        "Gemma smoke test hâlâ boş çıktı üretti. "
        "30 soruluk run'a geçmeyin."
    )

print("GEMMA SMOKE TEST BAŞARILI")

del model_smoke, frontend_smoke
clear_gpu()

## 10. Gemma'nın 30 soruluk run'ını çalıştır

Önce 9B smoke test başarılı olmalıdır.

Qwen ve Phi'yi yeniden çalıştırmayın.

Gemma için:

```python
MODEL_INDEX_TO_RUN = 2
OVERWRITE_MODEL_RUN = False
```

kullanın.

Yeni dosya `_nf4fp32` etiketi taşıyacak; eski boş Gemma FP16 run'ı artık okunmayacaktır.

In [ ]:
MODEL_INDEX_TO_RUN = 0
OVERWRITE_MODEL_RUN = False

candidate = CANDIDATE_MODELS[MODEL_INDEX_TO_RUN]
short_name = candidate["short_name"]
model_revision = candidate["revision"]
revision_short = model_revision[:8]
precision_tag = candidate["run_precision_tag"]

run_dir = RUNS_DIR / VALIDATED_DATASET_VERSION
run_dir.mkdir(parents=True, exist_ok=True)

output_path = run_dir / f"{short_name}_{revision_short}{precision_tag}_{PROMPT_VERSION}_{GENERATION_CONFIG_VERSION}.jsonl"
manifest_path = run_dir / f"{short_name}_{revision_short}{precision_tag}_{PROMPT_VERSION}_{GENERATION_CONFIG_VERSION}_manifest.json"

if output_path.exists() and not OVERWRITE_MODEL_RUN:
    print("Run zaten mevcut; tekrar çalıştırılmadı:", output_path)
else:
    frontend, model = load_model(candidate)
    torch.cuda.reset_peak_memory_stats()

    rows = []

    for i, row in sanity_df.iterrows():
        result = run_one(
            frontend,
            model,
            candidate,
            row["question"],
        )

        item = {
            "dataset_version": VALIDATED_DATASET_VERSION,
            "dataset_sha256": dataset_sha256,
            "question_id": row["question_id"],
            "question": row["question"],
            "reference_answer": row["reference_answer"],
            "accepted_aliases": row["accepted_aliases"],
            "answer_type": row["answer_type"],
            "category": row["category"],
            "model_id": candidate["model_id"],
            "model_revision": model_revision,\n        "backend": candidate["backend"],
            "model_short_name": short_name,\n            "model_backend": candidate["backend"],
            "prompt_version": PROMPT_VERSION,
            "generation_config_version": GENERATION_CONFIG_VERSION,
            **result,
        }

        item["automatic_alias_match"] = auto_alias_match(
            result["answer"],
            row["accepted_aliases"],
        )
        item["unknown_response"] = unknown_response(result["answer"])
        item["pure_unknown_response"] = pure_unknown_response(result["answer"])
        item["contains_unknown_marker"] = contains_unknown_marker(result["answer"])
        item["mixed_answer_unknown"] = mixed_answer_unknown(result["answer"])

        rows.append(item)

        with open(output_path, "w", encoding="utf-8") as f:
            for x in rows:
                f.write(json.dumps(x, ensure_ascii=False) + "\n")

        print(
            f"[{i+1:02d}/{len(sanity_df)}] "
            f"{row['question_id']} | {result['answer']!r} | "
            f"{result['generation_seconds']:.2f}s"
        )

    peak_gpu = torch.cuda.max_memory_allocated() / (1024 ** 3)

    run_manifest = {
        "purpose": "model_selection_sanity",
        "dataset_version": VALIDATED_DATASET_VERSION,
        "dataset_sha256": dataset_sha256,
        "sanity_split_file": SANITY_PATH.name,
        "model_id": candidate["model_id"],
        "model_revision": model_revision,\n        "backend": candidate["backend"],
        "trust_remote_code": candidate["trust_remote_code"],
        "prompt_version": PROMPT_VERSION,
        "generation_config_version": GENERATION_CONFIG_VERSION,
        "max_new_tokens": MAX_NEW_TOKENS,
        "do_sample": False,
        "quantization": "4bit_nf4_double_quant_fp16_compute",
        "gpu": torch.cuda.get_device_name(0),
        "peak_gpu_memory_gb": peak_gpu,
        "python": platform.python_version(),
        "pytorch": torch.__version__,
        "transformers": transformers.__version__,
        "created_at_utc": datetime.now(timezone.utc).isoformat(),
    }

    with open(manifest_path, "w", encoding="utf-8") as f:
        json.dump(run_manifest, f, ensure_ascii=False, indent=2)

    print("Run tamamlandı:", output_path)
    print("Peak GPU GB:", peak_gpu)

    del model, frontend
    clear_gpu()

## 11. Mevcut model run'larını birleştir

In [ ]:
def read_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

all_rows = []

for candidate in CANDIDATE_MODELS:
    path = (
        RUNS_DIR /
        VALIDATED_DATASET_VERSION /
        f"{candidate['short_name']}_{candidate['revision'][:8]}{candidate['run_precision_tag']}_{PROMPT_VERSION}_{GENERATION_CONFIG_VERSION}.jsonl"
    )
    if path.exists():
        all_rows.extend(read_jsonl(path))
    else:
        print("Henüz run yok:", candidate["model_id"])

combined_df = pd.DataFrame(all_rows)

if combined_df.empty:
    raise RuntimeError("Hiç model run'ı bulunamadı.")

# ------------------------------------------------------------------
# V4: Derived unknown flags are ALWAYS recomputed from raw answer text.
# Old JSONL files therefore do not need to be regenerated.
# ------------------------------------------------------------------
combined_df["pure_unknown_response"] = combined_df["answer"].apply(
    pure_unknown_response
)

combined_df["contains_unknown_marker"] = combined_df["answer"].apply(
    contains_unknown_marker
)

combined_df["mixed_answer_unknown"] = combined_df["answer"].apply(
    mixed_answer_unknown
)

# Backward-compatible field:
combined_df["unknown_response"] = combined_df[
    "pure_unknown_response"
]

print("Toplam model-answer:", len(combined_df))
display(
    combined_df[
        [
            "question_id",
            "model_id",
            "answer",
            "automatic_alias_match",
            "pure_unknown_response",
            "contains_unknown_marker",
            "mixed_answer_unknown",
            "generation_seconds",
            "mean_logprob_probe",
            "mean_token_entropy_probe",
        ]
    ].head(20)
)

## Unknown / uncertainty output taxonomy

Explicit `Bilmiyorum` davranışı üç ayrı sinyal olarak tutulur:

| Output | pure_unknown | contains_unknown | mixed_answer_unknown |
|---|---:|---:|---:|
| `Bilmiyorum.` | 1 | 1 | 0 |
| `Canberra.` | 0 | 0 | 0 |
| `Canberra. Bilmiyorum.` | 0 | 1 | 1 |

### Neden ayırıyoruz?

`Bilmiyorum.` saf bir abstention/unknown davranışıdır.

`Canberra. Bilmiyorum.` ise aynı output içinde hem substantive answer hem explicit uncertainty
bulundurduğu için farklı bir model davranışıdır. Bu output post-processing ile kesilmez.

Ana backward-compatible `unknown_response` alanı `pure_unknown_response` ile aynı tutulur.
`mixed_answer_unknown` ayrı feature/hata analizi sinyali olarak saklanır.

### Model-selection correctness kuralı

Strict kısa factual QA protokolünde `Canberra. Bilmiyorum.` gibi kendi içinde çelişkili mixed
cevaplar `manual_is_correct = 0` olarak etiketlenir. Bu kural bütün modeller için aynıdır.

In [ ]:
unknown_diagnostics = (
    combined_df
    .groupby("model_id")
    .agg(
        n=("question_id", "count"),
        pure_unknown_n=("pure_unknown_response", "sum"),
        contains_unknown_n=("contains_unknown_marker", "sum"),
        mixed_answer_unknown_n=("mixed_answer_unknown", "sum"),
    )
    .reset_index()
)

unknown_diagnostics["pure_unknown_rate"] = (
    unknown_diagnostics["pure_unknown_n"]
    / unknown_diagnostics["n"]
)

unknown_diagnostics["contains_unknown_rate"] = (
    unknown_diagnostics["contains_unknown_n"]
    / unknown_diagnostics["n"]
)

unknown_diagnostics["mixed_answer_unknown_rate"] = (
    unknown_diagnostics["mixed_answer_unknown_n"]
    / unknown_diagnostics["n"]
)

display(unknown_diagnostics)

print("\nMixed answer + unknown outputs:")
display(
    combined_df.loc[
        combined_df["mixed_answer_unknown"] == 1,
        [
            "question_id",
            "model_id",
            "answer",
            "reference_answer",
        ],
    ]
)

## 12. Manuel model-selection review workbook

`automatic_alias_match` yalnız yardımcı kontroldür.

Her cevap için:
- `manual_is_correct = 1 / 0`
- `manual_notes`

doldurulur.

Workbook yeniden oluşturulursa mevcut manuel etiketler korunur.

In [ ]:
REVIEW_PATH = RESULTS_DIR / (
    f"{VALIDATED_DATASET_VERSION}_model_selection_review_{GENERATION_CONFIG_VERSION}.xlsx"
)

work = combined_df.copy()
work["accepted_aliases"] = work["accepted_aliases"].apply(
    lambda x: " | ".join(x) if isinstance(x, list) else str(x)
)
work["manual_is_correct"] = ""
work["manual_notes"] = ""

key_cols = [
    "dataset_version",
    "question_id",
    "model_id",
    "model_revision",
    "prompt_version",
    "generation_config_version",
]

if REVIEW_PATH.exists():
    old = pd.read_excel(REVIEW_PATH)

    old_lookup = {
        tuple(str(row[c]) for c in key_cols): row
        for _, row in old.iterrows()
    }

    for idx, row in work.iterrows():
        key = tuple(str(row[c]) for c in key_cols)
        if key in old_lookup:
            prev = old_lookup[key]
            work.at[idx, "manual_is_correct"] = prev.get(
                "manual_is_correct", ""
            )
            work.at[idx, "manual_notes"] = prev.get(
                "manual_notes", ""
            )

cols = [
    "dataset_version",
    "question_id",
    "question",
    "reference_answer",
    "accepted_aliases",
    "answer_type",
    "category",
    "model_id",
    "model_revision",
    "answer",
    "automatic_alias_match",
    "pure_unknown_response",
    "contains_unknown_marker",
    "mixed_answer_unknown",
    "unknown_response",
    "content_token_count",
    "generation_seconds",
    "mean_logprob_probe",
    "min_logprob_probe",
    "mean_token_entropy_probe",
    "logprob_probe_available",
    "entropy_probe_available",
    "prompt_version",
    "generation_config_version",
    "manual_is_correct",
    "manual_notes",
]

work[cols].sort_values(
    ["question_id", "model_id"]
).to_excel(REVIEW_PATH, index=False)

print(REVIEW_PATH)

## 13. Manuel etiketlerden model summary üret

In [ ]:
review = pd.read_excel(REVIEW_PATH)

review["manual_is_correct"] = pd.to_numeric(
    review["manual_is_correct"],
    errors="coerce",
)

missing = review["manual_is_correct"].isna()

if missing.any():
    print("Eksik etiket:", int(missing.sum()))
    display(
        review.loc[
            missing,
            ["question_id", "model_id", "answer", "reference_answer"]
        ].head(20)
    )
    raise ValueError("Önce tüm manual_is_correct alanlarını 0/1 doldurun.")

if not review["manual_is_correct"].isin([0, 1]).all():
    raise ValueError("manual_is_correct yalnız 0/1 olabilir.")

summary = (
    review.groupby("model_id")
    .agg(
        n=("question_id", "count"),
        manual_accuracy=("manual_is_correct", "mean"),
        auto_alias_match_rate=("automatic_alias_match", "mean"),
        pure_unknown_rate=("pure_unknown_response", "mean"),
        contains_unknown_rate=("contains_unknown_marker", "mean"),
        mixed_answer_unknown_rate=("mixed_answer_unknown", "mean"),
        mean_generation_seconds=("generation_seconds", "mean"),
        median_generation_seconds=("generation_seconds", "median"),
        mean_output_tokens=("content_token_count", "mean"),
        logprob_probe_success_rate=("logprob_probe_available", "mean"),
        entropy_probe_success_rate=("entropy_probe_available", "mean"),
    )
    .reset_index()
    .sort_values("manual_accuracy", ascending=False)
)

display(summary)

SUMMARY_PATH = RESULTS_DIR / (
    f"{VALIDATED_DATASET_VERSION}_model_selection_summary_{GENERATION_CONFIG_VERSION}.csv"
)

summary.to_csv(SUMMARY_PATH, index=False)
print(SUMMARY_PATH)

## 14. Üç ana modeli dondur

Tez protokolü en az üç model karşılaştırması gerektirdiği için üç teknik olarak uygun model de
ana deneylere dahil edilir:

1. `google/gemma-3-4b-it`
2. `Qwen/Qwen3-4B`
3. `microsoft/Phi-4-mini-instruct`

30 soruluk sanity sonuçları **nihai performans tahmini değildir**; yalnız screening/model-selection
girdisidir.

Üç modelin de:

- deterministic generation
- token log-probability extraction
- token entropy extraction
- T4 üzerinde 4-bit inference

koşullarını sağladığı doğrulanmıştır.

Model güçlerinin farklı olması calibration çalışması açısından korunur; modeller accuracy'leri
eşitlemek amacıyla seçilmez veya elenmez.

Bu hücre çalıştırıldığında `selected_models_v1.json` immutable kabul edilir.

In [ ]:
# ============================================================
# ANA TEZ MODELLERİ
# ============================================================
SELECTED_MODELS = [
    "google/gemma-3-4b-it",
    "Qwen/Qwen3-4B",
    "microsoft/Phi-4-mini-instruct",
]
# ============================================================

REQUIRED_MAIN_MODEL_COUNT = 3

if len(SELECTED_MODELS) < REQUIRED_MAIN_MODEL_COUNT:
    raise ValueError(
        f"Tez protokolü en az {REQUIRED_MAIN_MODEL_COUNT} ana model gerektiriyor."
    )

valid_ids = {
    x["model_id"]
    for x in CANDIDATE_MODELS
}

if not set(SELECTED_MODELS).issubset(
    valid_ids
):
    raise ValueError(
        "Candidate listesinde olmayan model seçildi."
    )

# Summary'nin gerçekten üç modeli içerdiğini doğrula.
summary_check = pd.read_csv(
    SUMMARY_PATH
)

missing_summary_models = (
    set(SELECTED_MODELS)
    - set(summary_check["model_id"])
)

if missing_summary_models:
    raise RuntimeError(
        "Aşağıdaki seçili modeller için model-selection summary yok: "
        f"{sorted(missing_summary_models)}"
    )

if len(summary_check) < REQUIRED_MAIN_MODEL_COUNT:
    raise RuntimeError(
        "Üç modelin screening sonucu tamamlanmadan seçim dondurulamaz."
    )

# Exact model specs
selected_specs = []

for model_id in SELECTED_MODELS:
    candidate = next(
        x
        for x in CANDIDATE_MODELS
        if x["model_id"] == model_id
    )

    selected_specs.append({
        "model_id": candidate["model_id"],
        "short_name": candidate["short_name"],
        "revision": candidate["revision"],
        "backend": candidate["backend"],
        "trust_remote_code": candidate["trust_remote_code"],
        "disable_thinking": candidate["disable_thinking"],
        "compute_dtype": candidate["compute_dtype"],
        "model_dtype": candidate["model_dtype"],
        "run_precision_tag": candidate["run_precision_tag"],
    })

# Screening metrics are copied into manifest for provenance only.
screening_records = (
    summary_check[
        summary_check["model_id"].isin(
            SELECTED_MODELS
        )
    ]
    .to_dict(orient="records")
)

selection_path = (
    RESULTS_DIR /
    "selected_models_v1.json"
)

if selection_path.exists():
    raise FileExistsError(
        "Selection manifest zaten mevcut ve immutable kabul ediliyor:\n"
        f"{selection_path}"
    )

selection = {
    "selection_version": "model_selection_v1",
    "purpose": "three_main_models_for_thesis",
    "required_main_model_count": REQUIRED_MAIN_MODEL_COUNT,

    "dataset_version": VALIDATED_DATASET_VERSION,
    "dataset_sha256": dataset_sha256,

    "sanity_split_file": SANITY_PATH.name,
    "sanity_question_ids": sanity_df[
        "question_id"
    ].tolist(),
    "sanity_questions_must_be_excluded_from_final_test": True,

    "prompt_version": PROMPT_VERSION,
    "generation_config_version": GENERATION_CONFIG_VERSION,

    "selection_policy": {
        "screening_is_not_final_performance_estimate": True,
        "models_not_selected_to_equalize_accuracy": True,
        "all_models_required_to_support_logprob": True,
        "all_models_required_to_support_entropy": True,
        "minimum_main_models": 3,
    },

    "selected_models": selected_specs,

    "screening_summary": screening_records,

    "summary_file": SUMMARY_PATH.name,
    "review_file": REVIEW_PATH.name,

    "selected_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
}

with open(
    selection_path,
    "x",
    encoding="utf-8",
) as f:
    json.dump(
        selection,
        f,
        ensure_ascii=False,
        indent=2,
    )

print("ÜÇ ANA MODEL DONDURULDU")
print(selection_path)

print("\nSelected models:")
for spec in selected_specs:
    print(
        "-",
        spec["model_id"],
        "| revision:",
        spec["revision"][:12],
        "| compute:",
        spec["compute_dtype"],
    )

# Notebook 03 tamamlanma kriterleri

- [ ] `validated_vXXX` checksum doğrulandı
- [ ] 30 soruluk immutable development sanity subset üretildi
- [ ] Bu ID'ler final test dışı olarak kaydedildi
- [ ] Qwen3-4B aynı deterministic contract ile çalıştırıldı
- [ ] Phi-4-mini aynı deterministic contract ile çalıştırıldı
- [ ] Gemma-3-4B-it exact revision SHA ile çalıştırıldı
- [ ] Üç modelde token logprob extraction %100 başarılı
- [ ] Üç modelde entropy extraction %100 başarılı
- [ ] Unknown/mixed-output sinyalleri hesaplandı
- [ ] 90 model-answer satırı manuel `manual_is_correct` ile etiketlendi
- [ ] Üç modelin screening summary'si üretildi
- [ ] Üç model ana tez modelleri olarak donduruldu
- [ ] `selected_models_v1.json` oluşturuldu

## Ana tez modelleri

```text
Gemma-3-4B-it
Qwen3-4B
Phi-4-mini-instruct
```

## Sonraki aşama

**04 — Main Answer Generation & Internal Features**

04 notebook:

- `selected_models_v1.json` dosyasını okuyacak,
- üç modelin exact revision/runtime config'ini kullanacak,
- açık bir `validated_vXXX` sürümü üzerinde incremental inference yapacak,
- her `question_id × model_id × config` kombinasyonunu cache'leyecek,
- deterministic ana cevap ile internal confidence feature'ları kalıcı olarak üretecek.